In [1]:
import torch
import torch.utils.data
import torchvision
from NssMPC.application.neural_network.utils.converter import load_model
from NssMPC.application.neural_network.utils.converter import share_model
from NssMPC.application.neural_network.utils.converter import share_data

In [6]:
# training CNN
exec(open('/home/adslab/code/ADSMPC-python/NssMPClib/data/CNN/CNN_MNIST_train.py').read())

Start Training!
[1, 46900] loss:0.2820
[2, 46900] loss:0.2097
[3, 46900] loss:0.1220
[4, 46900] loss:0.2537
[5, 46900] loss:0.1700
Finished Training
time 0.4564089775085449
Accuracy of the network on the 100 test images:95.32%


In [3]:
from data.CNN.CNN import CNN
from NssMPC.application.neural_network.party.neural_network_party import NeuralNetworkCS

In [4]:
import threading

# set Server
server = NeuralNetworkCS(type='server')

def set_server():
    # CS connect

    # server.connect(server_server_address, server_client_address, client_server_address, client_client_address)
    server.online()

# set Client
client = NeuralNetworkCS(type='client')

def set_client():
    # CS connect

    # client.connect(client_server_address, client_client_address, server_server_address, server_client_address)
    client.online()

server_thread = threading.Thread(target=set_server)
client_thread = threading.Thread(target=set_client)

server_thread.start()
client_thread.start()
client_thread.join()
server_thread.join()

TCPServer waiting for connection ......
TCPServer waiting for connection ......
successfully connect to server 127.0.0.1:8000
TCPServer successfully connected by :('127.0.0.1', 9100)
successfully connect to server 127.0.0.1:9000
TCPServer successfully connected by :('127.0.0.1', 8200)


In [5]:
import torchvision.transforms as transforms
from torch.utils.data import Subset
from NssMPC.config import NN_path, DEVICE
from NssMPC.config.runtime import PartyRuntime

def server_predict():
    with PartyRuntime(server):
        net = CNN()

        net.load_state_dict(torch.load('/home/adslab/code/ADSMPC-python/NssMPClib/data/CNN/CNN_MNIST.pkl'))
        shared_param, shared_param_for_other = share_model(net)
        server.send(shared_param_for_other)

        num = server.dummy_model(net)
        net = load_model(net, shared_param)
        while num:
            shared_data = server.receive()
            server.inference(net, shared_data)
            num -= 1
    # close party after inference
    server.close()

def client_predict():
    with PartyRuntime(client):
        net = CNN()
        transform1 = transforms.Compose([transforms.ToTensor()])
        test_set = torchvision.datasets.MNIST(root=NN_path, train=False, download=True, transform=transform1)
        indices = list(range(5))
        subset_data = Subset(test_set, indices)
        test_loader = torch.utils.data.DataLoader(subset_data, batch_size=1, shuffle=False, num_workers=0)

        shared_param = client.receive()
        num = client.dummy_model(test_loader)
        net = load_model(net, shared_param)

        for data in test_loader:
            correct = 0
            total = 0
            images, labels = data

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            shared_data, shared_data_for_other = share_data(images)
            client.send(shared_data_for_other)

            res = client.inference(net, shared_data)

            _, predicted = torch.max(res, 1)

            print("Predicted result is: ", predicted)

        # close party after inference
    client.close()

server_thread = threading.Thread(target=server_predict)
client_thread = threading.Thread(target=client_predict)

server_thread.start()
client_thread.start()
client_thread.join()
server_thread.join()

secrelu
secrelu
secrelu
secrelu
Predicted result is:  tensor([7], device='cuda:0')
secrelu
secrelu
secrelu
secrelu
Predicted result is:  tensor([2], device='cuda:0')
secrelu
secrelu
secrelu
secrelu
Predicted result is:  tensor([1], device='cuda:0')
secrelu
secrelu
secrelu
secrelu
Predicted result is:  tensor([0], device='cuda:0')
secrelu
secrelu
secrelu
secrelu
Predicted result is:  tensor([4], device='cuda:0')
Communicator: 127.0.0.1 closed.Communicator: 127.0.0.1 closed.
Communication costs:
	send rounds: 77		send bytes: 12.034564971923828 MB.
	recv rounds: 52		recv bytes: 10.003826141357422 MB.

Communication costs:
	send rounds: 52		send bytes: 10.003826141357422 MB.
	recv rounds: 77		recv bytes: 12.034564971923828 MB.
